# Cafe Sales — Batch Cleaning & KPI Pipeline

**Session 04 — cafe sales exercise (batch)**
Author: Khushbu Shobhashna

**Dataset:** Kaggle `ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training` — `dirty_cafe_sales.csv`

---

### Why batch here

This is the deliberate counterpart to the e-commerce notebook, which uses Structured Streaming
with Auto Loader. The cafe file is a single static export — streaming it would be a costume over
a batch job. Batch is the honest choice, and it frees the notebook to use techniques that are
awkward or illegal in a streaming query: window functions, `countDistinct`, self-joins,
percentile aggregates, and full-history ranking.

| Exercise | Engine | Techniques it demonstrates |
|---|---|---|
| E-commerce orders | Structured Streaming | Auto Loader, checkpoints, watermarks, batch/stream parity |
| **Cafe sales (this one)** | **Batch** | window functions, Pareto/ABC, rank correlation, null co-occurrence |

---

### The dirt in this file

| Column group | Junk |
|---|---|
| `Quantity`, `Price Per Unit`, `Total Spent` | literal `"ERROR"` and true nulls |
| `Payment Method`, `Location` | literal `"UNKNOWN"` and true nulls |
| `Transaction Date` | malformed / unparseable strings |

Unlike the e-commerce dataset, this dirt is **injected** rather than organic — someone wrote
`"ERROR"` into cells on purpose. That makes it a cleaner teaching set but a less realistic one,
which is worth saying out loud rather than pretending otherwise.

The one genuinely interesting property: `Total Spent = Quantity x Price Per Unit`, so any single
missing value in that triple is **arithmetically recoverable**. Most of the "missing" data here
is not actually lost, and a pipeline that drops those rows throws away recoverable revenue.

## 0. Imports and configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

In [0]:
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA  = "cafe_sales"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.data")

VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/data"
dbutils.fs.mkdirs(VOLUME)

print(f"Catalog/schema : {CATALOG}.{SCHEMA}")
print(f"Data volume    : {VOLUME}")

## 1. Load the source data

Put `dirty_cafe_sales.csv` next to this notebook. If it is absent, a synthetic generator produces
a file with the same schema and the same injected dirt so the notebook runs end to end.

Everything is written into the UC volume — serverless compute has no writable local filesystem
outside `/Workspace`.

In [0]:
import os, csv, random, datetime

SOURCE_CSV = os.path.join(os.getcwd(), "dirty_cafe_sales.csv")
USING_SYNTHETIC = not os.path.exists(SOURCE_CSV)

HEADER = ["Transaction ID", "Item", "Quantity", "Price Per Unit",
          "Total Spent", "Payment Method", "Location", "Transaction Date"]

if USING_SYNTHETIC:
    print("!! dirty_cafe_sales.csv not found — generating SYNTHETIC data with the same dirt.")
    print("!! Download ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training to use the real file.\n")

    random.seed(7)
    MENU = [("Coffee", 2.0), ("Tea", 1.5), ("Sandwich", 4.0), ("Salad", 5.0),
            ("Cake", 3.0), ("Cookie", 1.0), ("Smoothie", 4.0), ("Juice", 3.0)]
    PAYMENTS  = ["Cash", "Credit Card", "Digital Wallet"]
    LOCATIONS = ["In-store", "Takeaway"]

    def dirty(value, sentinel, rate):
        """Inject a sentinel or a blank at the given rate."""
        r = random.random()
        if r < rate * 0.6:
            return sentinel
        if r < rate:
            return ""
        return value

    rows = []
    start = datetime.date(2023, 1, 1)
    for i in range(10000):
        item, unit_price = random.choice(MENU)
        qty = random.randint(1, 5)
        total = round(qty * unit_price, 2)
        day = start + datetime.timedelta(days=random.randint(0, 364))

        # Dates: mostly valid ISO, sometimes malformed, sometimes blank.
        r = random.random()
        if r < 0.04:
            date_str = random.choice([f"{day.day}-{day.month}-{day.year}", "not a date",
                                      f"{day.year}/{day.month}/{day.day}", "13/45/2023"])
        elif r < 0.07:
            date_str = ""
        else:
            date_str = day.isoformat()

        rows.append([
            f"TXN_{1000000 + i}",
            dirty(item, "UNKNOWN", 0.03),
            dirty(str(qty), "ERROR", 0.05),
            dirty(f"{unit_price:.2f}", "ERROR", 0.05),
            dirty(f"{total:.2f}", "ERROR", 0.08),
            dirty(random.choice(PAYMENTS), "UNKNOWN", 0.10),
            dirty(random.choice(LOCATIONS), "UNKNOWN", 0.10),
            date_str,
        ])

    SOURCE_CSV = f"{VOLUME}/dirty_cafe_sales_synthetic.csv"
    with open(SOURCE_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(HEADER)
        w.writerows(rows)
    print(f"Generated {len(rows):,} rows -> {SOURCE_CSV}")
else:
    print(f"Using real dataset: {SOURCE_CSV}")

In [0]:
SPARK_PATH = SOURCE_CSV if SOURCE_CSV.startswith("/Volumes") else f"file:{SOURCE_CSV}"

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)   # dirty values break inference — read as string, cast later
    .csv(SPARK_PATH)
)

SOURCE_COLS = raw.columns
TOTAL_ROWS  = raw.count()

print(f"Rows: {TOTAL_ROWS:,}   Columns: {len(SOURCE_COLS)}")
raw.printSchema()
display(raw.limit(20))

## 2. Profile before cleaning

The sentinel list below is derived from what is actually in the file, not guessed. If a future
version of the dataset introduces a new junk value, this cell surfaces it instead of letting it
slip through as a legitimate category.

In [0]:
SENTINELS = ["ERROR", "UNKNOWN", "NaN", "nan", "", "NULL", "null", "None", "N/A", "-"]

profile = []
for c in SOURCE_COLS:
    hits = (
        raw.select(F.trim(F.col(f"`{c}`")).alias("v"))
        .filter(F.col("v").isNull() | F.col("v").isin(SENTINELS))
        .groupBy("v").count().collect()
    )
    for h in hits:
        profile.append((c, h["v"] if h["v"] is not None else "<null>", h["count"]))

display(
    spark.createDataFrame(profile, ["Column", "Junk_Value", "Count"])
    .withColumn("Pct_Of_Rows", F.round(F.col("Count") / F.lit(TOTAL_ROWS) * 100, 2))
    .orderBy(F.desc("Count"))
)

### Are the junk values independent, or do they cluster?

A question the profile above cannot answer. If corruption clusters on the same rows, a small
number of rows are badly broken and the rest are fine — a very different remediation than the
same volume of junk spread thinly across every row.

In [0]:
junk_flags = raw
for c in SOURCE_COLS:
    junk_flags = junk_flags.withColumn(
        f"bad__{c}",
        (F.col(f"`{c}`").isNull() | F.trim(F.col(f"`{c}`")).isin(SENTINELS)).cast("int"),
    )

junk_flags = junk_flags.withColumn(
    "bad_field_count", sum(F.col(f"bad__{c}") for c in SOURCE_COLS)
)

display(
    junk_flags.groupBy("bad_field_count").count()
    .withColumn("Pct_Of_Rows", F.round(F.col("count") / F.lit(TOTAL_ROWS) * 100, 2))
    .orderBy("bad_field_count")
)

## 3. Clean

Order matters. Sentinels become NULL **before** casting — cast first and `"ERROR"` silently
becomes null, destroying the distinction between "someone recorded an error" and "nothing was
recorded at all".

In [0]:
cleaned = raw
for c in SOURCE_COLS:
    cleaned = cleaned.withColumn(
        c,
        F.when(
            F.col(f"`{c}`").isNull() | F.trim(F.col(f"`{c}`")).isin(SENTINELS),
            F.lit(None).cast("string"),
        ).otherwise(F.trim(F.col(f"`{c}`"))),
    )

cleaned = (
    cleaned
    .withColumn("Quantity", F.col("Quantity").cast("double"))
    .withColumn("Price Per Unit", F.col("`Price Per Unit`").cast("double"))
    .withColumn("Total Spent", F.col("`Total Spent`").cast("double"))
)
cleaned.printSchema()

### Recover what arithmetic allows

`Total Spent = Quantity x Price Per Unit`. With any two of the three present, the third is
recoverable exactly — this is not imputation by average or by guess, it is algebra. Each
recovery is flagged so a sceptical reader can exclude them and see whether the KPIs move.

In [0]:
q, p, t = F.col("Quantity"), F.col("`Price Per Unit`"), F.col("`Total Spent`")

cleaned = (
    cleaned
    .withColumn("recovered_total",    t.isNull() & q.isNotNull() & p.isNotNull())
    .withColumn("recovered_quantity", q.isNull() & t.isNotNull() & p.isNotNull() & (p != 0))
    .withColumn("recovered_price",    p.isNull() & t.isNotNull() & q.isNotNull() & (q != 0))
)

cleaned = (
    cleaned
    .withColumn("Total Spent",    F.when(F.col("recovered_total"),    F.round(q * p, 2)).otherwise(t))
    .withColumn("Quantity",       F.when(F.col("recovered_quantity"), F.round(t / p, 2)).otherwise(q))
    .withColumn("Price Per Unit", F.when(F.col("recovered_price"),    F.round(t / q, 2)).otherwise(p))
)

cleaned = cleaned.withColumn(
    "was_recovered",
    F.col("recovered_total") | F.col("recovered_quantity") | F.col("recovered_price"),
)

display(cleaned.select(
    F.sum(F.col("recovered_total").cast("int")).alias("Total_Recovered"),
    F.sum(F.col("recovered_quantity").cast("int")).alias("Quantity_Recovered"),
    F.sum(F.col("recovered_price").cast("int")).alias("Price_Recovered"),
    F.sum(F.col("was_recovered").cast("int")).alias("Rows_Touched"),
))

### Dates, and the difference between malformed and absent

`to_date` returns null for both a blank string and `"13/45/2023"`, which collapses two different
problems into one number. Splitting them matters: blanks mean the field was never captured,
malformed values mean it was captured through a broken path.

In [0]:
cleaned = (
    cleaned
    # try_to_date is used instead of to_date because ANSI mode (default on
    # serverless) makes to_date throw an error on unparseable input.
    # try_to_date returns NULL, allowing us to classify malformed dates.
    .withColumn(
        "txn_date",
        F.expr("try_to_date(`Transaction Date`, 'yyyy-MM-dd')")
    )
    .withColumn(
        "date_missing",
        F.col("`Transaction Date`").isNull()
    )
    .withColumn(
        "date_malformed",
        F.col("`Transaction Date`").isNotNull()
        & F.col("txn_date").isNull()
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("txn_date")
    )
    .withColumn(
        "day_name",
        F.date_format("txn_date", "EEEE")
    )
    .withColumn(
        "day_type",
        # Spark dayofweek: 1 = Sunday ... 7 = Saturday.
        # NULL when the date cannot be parsed, so unknown dates are not
        # incorrectly classified as "Weekday".
        F.when(
            F.col("txn_date").isNull(),
            F.lit(None).cast("string")
        )
        .when(
            F.dayofweek("txn_date").isin(1, 7),
            F.lit("Weekend")
        )
        .otherwise(F.lit("Weekday"))
    )
    .withColumn(
        "year_month",
        F.date_format("txn_date", "yyyy-MM")
    )
)

### Transaction validity

A transaction is **complete** when every field a KPI might need survived. Kept as a single
boolean so each KPI can decide whether it cares.

In [0]:
cleaned = cleaned.withColumn(
    "is_complete",
    F.col("Quantity").isNotNull()
    & F.col("`Total Spent`").isNotNull()
    & F.col("`Price Per Unit`").isNotNull()
    & F.col("Item").isNotNull()
    & F.col("`Payment Method`").isNotNull()
    & F.col("Location").isNotNull()
    & F.col("txn_date").isNotNull(),
)

# Revenue-usable is a weaker bar than complete: a sale with an unknown payment method
# still counts toward revenue. Conflating the two understates every revenue KPI.
cleaned = cleaned.withColumn(
    "is_revenue_usable",
    F.col("`Total Spent`").isNotNull() & (F.col("`Total Spent`") > 0),
)

print(f"Complete transactions      : {cleaned.filter('is_complete').count():,}")
print(f"Revenue-usable transactions: {cleaned.filter('is_revenue_usable').count():,}")
print(f"Total rows                 : {TOTAL_ROWS:,}")

## 4. Persist to Delta

In [0]:
persisted = cleaned
for c in persisted.columns:
    persisted = persisted.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))

(persisted.write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable("cafe_sales_cleaned"))

print("Saved table: cafe_sales_cleaned")

# No .cache() — serverless does not support PERSIST and manages its own caching.
df = cleaned
revenue = df.filter(F.col("is_revenue_usable"))
REVENUE_ROWS = revenue.count()
display(spark.table("cafe_sales_cleaned").limit(10))

---
# KPIs

Ten KPIs covering the same ground as the reference exercise, implemented independently. Where the
obvious one-line version would mislead, the extra column that makes it honest is included and the
reason is stated.

## KPI 1 — Revenue by Product

Revenue alone ranks products but does not explain them. Units and average line value separate
"sells constantly at low value" from "sells rarely at high value" — two products can post the
same revenue and need completely different decisions.

In [0]:
kpi_revenue_by_product = (
    revenue
    .filter(F.col("Item").isNotNull())
    .groupBy("Item")
    .agg(
        F.round(F.sum("`Total Spent`"), 2).alias("Revenue"),
        F.count("*").alias("Transactions"),
        F.round(F.sum("Quantity"), 0).cast("long").alias("Units"),
        F.round(F.avg("`Total Spent`"), 2).alias("Avg_Line_Value"),
        F.round(F.avg("`Price Per Unit`"), 2).alias("Avg_Unit_Price"),
    )
    .withColumn(
        "Revenue_Share_Pct",
        F.round(F.col("Revenue") / F.sum("Revenue").over(Window.partitionBy()) * 100, 2),
    )
    .orderBy(F.desc("Revenue"))
)
display(kpi_revenue_by_product)

## KPI 2 — Revenue by Location

`Location` is null for a meaningful share of rows. Those are reported as their own line rather
than dropped: silently excluding them makes the location split add up to less than total revenue,
and nobody notices until the numbers are questioned in a meeting.

In [0]:
kpi_revenue_by_location = (
    revenue
    .withColumn("Location_Label", F.coalesce(F.col("Location"), F.lit("(not recorded)")))
    .groupBy("Location_Label")
    .agg(
        F.round(F.sum("`Total Spent`"), 2).alias("Revenue"),
        F.count("*").alias("Transactions"),
        F.round(F.avg("`Total Spent`"), 2).alias("Avg_Transaction_Value"),
        F.round(F.avg("Quantity"), 2).alias("Avg_Items"),
    )
    .withColumn(
        "Revenue_Share_Pct",
        F.round(F.col("Revenue") / F.sum("Revenue").over(Window.partitionBy()) * 100, 2),
    )
    .orderBy(F.desc("Revenue"))
)
display(kpi_revenue_by_location)

reconciled = kpi_revenue_by_location.agg(F.sum("Revenue")).first()[0]
actual = revenue.agg(F.sum("`Total Spent`")).first()[0]
print(f"Reconciles to total revenue: {abs(reconciled - actual) < 0.01}  ({reconciled:,.2f} vs {actual:,.2f})")

## KPI 3 — Revenue by Payment Method

Same treatment as location, plus average transaction value — the operationally useful question is
usually not "which method earns most" (that tracks volume) but "do customers spend differently
depending on how they pay".

In [0]:
kpi_revenue_by_payment = (
    revenue
    .withColumn("Payment_Label", F.coalesce(F.col("`Payment Method`"), F.lit("(not recorded)")))
    .groupBy("Payment_Label")
    .agg(
        F.round(F.sum("`Total Spent`"), 2).alias("Revenue"),
        F.count("*").alias("Transactions"),
        F.round(F.avg("`Total Spent`"), 2).alias("Avg_Transaction_Value"),
        F.round(F.expr("percentile_approx(`Total Spent`, 0.5)"), 2).alias("Median_Transaction"),
    )
    .withColumn(
        "Txn_Share_Pct",
        F.round(F.col("Transactions") / F.sum("Transactions").over(Window.partitionBy()) * 100, 2),
    )
    .orderBy(F.desc("Revenue"))
)
display(kpi_revenue_by_payment)

## KPI 4 — Weekend vs Weekday

The headline two-row split is nearly useless on its own: a weekend is 2 days and a weekday block
is 5, so "weekdays earn more" is arithmetic, not insight. **Revenue per day** is the comparable
number, and the per-weekday breakdown underneath shows which days actually carry the week.

In [0]:
dated = revenue.filter(F.col("txn_date").isNotNull())

kpi_day_type = (
    dated.groupBy("day_type")
    .agg(
        F.round(F.sum("`Total Spent`"), 2).alias("Revenue"),
        F.count("*").alias("Transactions"),
        F.countDistinct("txn_date").alias("Days_Observed"),
        F.round(F.avg("`Total Spent`"), 2).alias("Avg_Transaction_Value"),
    )
    .withColumn("Revenue_Per_Day", F.round(F.col("Revenue") / F.col("Days_Observed"), 2))
    .orderBy("day_type")
)
display(kpi_day_type)

kpi_by_weekday = (
    dated.groupBy("day_of_week", "day_name")
    .agg(
        F.round(F.sum("`Total Spent`"), 2).alias("Revenue"),
        F.count("*").alias("Transactions"),
        F.countDistinct("txn_date").alias("Days_Observed"),
    )
    .withColumn("Revenue_Per_Day", F.round(F.col("Revenue") / F.col("Days_Observed"), 2))
    .orderBy("day_of_week")
)
display(kpi_by_weekday)

## KPI 5 — Peak Sales Day

A single peak date is fragile — one unusual day tells you little. A 7-day rolling average
alongside it separates a genuine trend from a spike, which is what a window function is for.

In [0]:
daily = (
    dated.groupBy("txn_date")
    .agg(
        F.round(F.sum("`Total Spent`"), 2).alias("Revenue"),
        F.count("*").alias("Transactions"),
    )
)

w7 = Window.orderBy("txn_date").rowsBetween(-6, 0)
daily_trend = (
    daily
    .withColumn("Rolling_7d_Avg", F.round(F.avg("Revenue").over(w7), 2))
    .withColumn("Vs_Rolling_Pct",
                F.round((F.col("Revenue") - F.col("Rolling_7d_Avg")) / F.col("Rolling_7d_Avg") * 100, 1))
    .orderBy("txn_date")
)
display(daily_trend)

print("Top 10 days by revenue:")
display(daily_trend.orderBy(F.desc("Revenue")).limit(10))

peak = daily.orderBy(F.desc("Revenue")).first()
print(f"Peak day: {peak['txn_date']} — {peak['Revenue']:,.2f} across {peak['Transactions']:,} transactions")

## KPI 6 — Best Selling Item by Quantity

Deliberately compared against the revenue ranking from KPI 1. When the two disagree, the cheap
high-volume items are subsidising attention while the revenue actually comes from elsewhere —
that disagreement is the finding, not either list on its own.

In [0]:
by_units = (
    revenue.filter(F.col("Item").isNotNull() & F.col("Quantity").isNotNull())
    .groupBy("Item")
    .agg(
        F.round(F.sum("Quantity"), 0).cast("long").alias("Units_Sold"),
        F.round(F.sum("`Total Spent`"), 2).alias("Revenue"),
    )
)

kpi_rank_comparison = (
    by_units
    .withColumn("Rank_By_Units",   F.rank().over(Window.orderBy(F.desc("Units_Sold"))))
    .withColumn("Rank_By_Revenue", F.rank().over(Window.orderBy(F.desc("Revenue"))))
    .withColumn("Rank_Gap", F.col("Rank_By_Revenue") - F.col("Rank_By_Units"))
    .orderBy("Rank_By_Units")
)
display(kpi_rank_comparison)

top_units = kpi_rank_comparison.orderBy("Rank_By_Units").first()
top_rev   = kpi_rank_comparison.orderBy("Rank_By_Revenue").first()
print(f"Best seller by volume : {top_units['Item']} ({top_units['Units_Sold']:,} units)")
print(f"Best seller by revenue: {top_rev['Item']} ({top_rev['Revenue']:,.2f})")
print("Same item?" , top_units["Item"] == top_rev["Item"])

## KPI 7 — Product Revenue Contribution

Share alone does not answer "how many products do I actually depend on". The cumulative column
does, and the ABC banding turns it into something actionable: A = the products making the first
80% of revenue, B = the next 15%, C = the long tail.

In [0]:
w_desc = Window.orderBy(F.desc("Revenue"))

kpi_contribution = (
    revenue.filter(F.col("Item").isNotNull())
    .groupBy("Item")
    .agg(F.sum("`Total Spent`").alias("Revenue"))
    .withColumn("Total_Revenue", F.sum("Revenue").over(Window.partitionBy()))
    .withColumn("Contribution_Pct", F.round(F.col("Revenue") / F.col("Total_Revenue") * 100, 2))
    .withColumn(
        "Cumulative_Pct",
        F.round(
            F.sum("Revenue").over(w_desc.rowsBetween(Window.unboundedPreceding, 0))
            / F.col("Total_Revenue") * 100, 2,
        ),
    )
    .withColumn(
        "ABC_Class",
        F.when(F.col("Cumulative_Pct") <= 80, "A")
         .when(F.col("Cumulative_Pct") <= 95, "B")
         .otherwise("C"),
    )
    .withColumn("Revenue", F.round("Revenue", 2))
    .drop("Total_Revenue")
    .orderBy(F.desc("Revenue"))
)
display(kpi_contribution)

display(
    kpi_contribution.groupBy("ABC_Class")
    .agg(F.count("*").alias("Products"), F.round(F.sum("Contribution_Pct"), 2).alias("Revenue_Pct"))
    .orderBy("ABC_Class")
)

## KPI 8 — Transaction Success Rate

A bare percentage is not actionable — it says something is wrong without saying what. The
breakdown underneath attributes every incomplete transaction to a cause.

⚠️ The cause counts **overlap**: one row can be missing both its payment method and its date, so
they sum to more than the incomplete total. Presenting them as a clean partition would be wrong.

In [0]:
complete_n   = df.filter(F.col("is_complete")).count()
incomplete_n = TOTAL_ROWS - complete_n

kpi_success = spark.createDataFrame(
    [(TOTAL_ROWS, complete_n, incomplete_n,
      round(complete_n / TOTAL_ROWS * 100, 2),
      REVENUE_ROWS, round(REVENUE_ROWS / TOTAL_ROWS * 100, 2))],
    ["Total_Rows", "Complete_Rows", "Incomplete_Rows",
     "Complete_Rate_Pct", "Revenue_Usable_Rows", "Revenue_Usable_Pct"],
)
display(kpi_success)

incomplete = df.filter(~F.col("is_complete"))
causes = incomplete.select(
    F.sum(F.col("Quantity").isNull().cast("int")).alias("Missing_Quantity"),
    F.sum(F.col("`Price Per Unit`").isNull().cast("int")).alias("Missing_Price"),
    F.sum(F.col("`Total Spent`").isNull().cast("int")).alias("Missing_Total"),
    F.sum(F.col("Item").isNull().cast("int")).alias("Missing_Item"),
    F.sum(F.col("`Payment Method`").isNull().cast("int")).alias("Missing_Payment"),
    F.sum(F.col("Location").isNull().cast("int")).alias("Missing_Location"),
    F.sum(F.col("txn_date").isNull().cast("int")).alias("Unusable_Date"),
).first().asDict()

display(
    spark.createDataFrame(
        [(k, int(v or 0), round((v or 0) / incomplete_n * 100, 2) if incomplete_n else None)
         for k, v in causes.items()],
        ["Cause", "Rows", "Pct_Of_Incomplete"],
    ).orderBy(F.desc("Rows"))
)
print(f"Causes overlap — they sum to more than the {incomplete_n:,} incomplete rows.")

## KPI 9 — Missing Data Rate

Three readings, because the single number people usually quote is ambiguous:

1. **cell-level** — the headline rate across all source cells
2. **per column** — where the problem actually is
3. **post-recovery** — what remains after the algebra in section 3, which is the number that
   should drive any remediation decision

In [0]:
def null_rates(frame, label):
    counts = frame.select(
        [F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c) for c in SOURCE_COLS]
    ).first().asDict()
    total_cells = TOTAL_ROWS * len(SOURCE_COLS)
    return counts, round(sum(counts.values()) / total_cells * 100, 2)

# Pre-recovery view: re-apply sentinel nulling to raw without the arithmetic recovery.
pre = raw
for c in SOURCE_COLS:
    pre = pre.withColumn(
        c,
        F.when(F.col(f"`{c}`").isNull() | F.trim(F.col(f"`{c}`")).isin(SENTINELS),
               F.lit(None).cast("string")).otherwise(F.trim(F.col(f"`{c}`"))),
    )

pre_counts,  pre_rate  = null_rates(pre, "before")
post_counts, post_rate = null_rates(df, "after")

print(f"Cell-level missing rate before recovery: {pre_rate}%")
print(f"Cell-level missing rate after recovery : {post_rate}%")
print(f"Recovered by arithmetic                : {round(pre_rate - post_rate, 2)} percentage points")

display(
    spark.createDataFrame(
        [(c, int(pre_counts[c] or 0), int(post_counts[c] or 0),
          round((pre_counts[c] or 0) / TOTAL_ROWS * 100, 2),
          round((post_counts[c] or 0) / TOTAL_ROWS * 100, 2))
         for c in SOURCE_COLS],
        ["Column", "Nulls_Before", "Nulls_After", "Rate_Before_Pct", "Rate_After_Pct"],
    ).orderBy(F.desc("Rate_After_Pct"))
)

## KPI 10 — Invalid Date Rate

Malformed and absent are reported separately. They have different causes and different fixes: a
malformed value means an upstream format or parser problem, a blank means the field was never
captured at all.

In [0]:
malformed_n = df.filter(F.col("date_malformed")).count()
missing_n   = df.filter(F.col("date_missing")).count()

pct = lambda n: round(n / TOTAL_ROWS * 100, 2) if TOTAL_ROWS else None

display(spark.createDataFrame(
    [(malformed_n, pct(malformed_n), missing_n, pct(missing_n),
      malformed_n + missing_n, pct(malformed_n + missing_n))],
    ["Malformed_Count", "Malformed_Pct", "Missing_Count", "Missing_Pct",
     "Total_Unusable", "Total_Unusable_Pct"],
))

print("Sample of the malformed values — these show which upstream format is leaking through:")
display(
    df.filter(F.col("date_malformed"))
    .groupBy("`Transaction Date`").count()
    .orderBy(F.desc("count")).limit(20)
)

revenue_at_risk = df.filter(F.col("txn_date").isNull() & F.col("is_revenue_usable")) \
                    .agg(F.sum("`Total Spent`")).first()[0] or 0.0
print(f"\nRevenue that cannot be placed on a date: {revenue_at_risk:,.2f}")

---
## Summary

In [0]:
total_revenue = revenue.agg(F.sum("`Total Spent`")).first()[0] or 0.0
a_class = kpi_contribution.filter(F.col("ABC_Class") == "A").count()

summary = [
    ("Data source",                 "SYNTHETIC (generated)" if USING_SYNTHETIC else "Kaggle dirty_cafe_sales"),
    ("Total rows",                  f"{TOTAL_ROWS:,}"),
    ("Complete transactions",       f"{complete_n:,} ({round(complete_n/TOTAL_ROWS*100,2)}%)"),
    ("Revenue-usable transactions", f"{REVENUE_ROWS:,} ({round(REVENUE_ROWS/TOTAL_ROWS*100,2)}%)"),
    ("Rows recovered by algebra",   f"{df.filter(F.col('was_recovered')).count():,}"),
    ("Total revenue",               f"{total_revenue:,.2f}"),
    ("Top product by revenue",      kpi_revenue_by_product.first()["Item"]),
    ("Best seller by volume",       top_units["Item"]),
    ("Products in class A",         f"{a_class} (first 80% of revenue)"),
    ("Peak day",                    str(peak["txn_date"])),
    ("Missing rate after recovery", f"{post_rate}%"),
    ("Unusable dates",              f"{pct(malformed_n + missing_n)}%"),
]
display(spark.createDataFrame(summary, ["Metric", "Value"]))

---
## Notes on the approach

**What this notebook does differently from a first-pass solution**

| Decision | Why |
|---|---|
| Sentinels nulled *before* casting | casting first destroys the difference between `"ERROR"` and blank |
| Missing values recovered by algebra, not averages | `Total = Qty x Price` makes them exactly recoverable; averaging would invent data |
| Every recovery flagged | lets a reader exclude recovered rows and check whether the KPIs move |
| `day_type` NULL for undatable rows | `.otherwise("Weekday")` would silently inflate the weekday bucket |
| Nulls shown as `(not recorded)`, not dropped | dropped groups make the split fail to reconcile to the total |
| Revenue-per-day in KPI 4 | 5 weekdays vs 2 weekend days makes raw totals incomparable |
| Volume rank compared against revenue rank | the disagreement between them is the actual finding |
| Overlapping failure causes labelled as overlapping | presenting them as a partition would be arithmetically wrong |
| Missing rate reported before *and* after recovery | the post-recovery number is the one that should drive decisions |

**Honest limitations**

- The dirt is injected, not organic — real cafe data would have subtler problems (timezone drift,
  duplicate transaction IDs, price changes over time) that this dataset does not contain.
- There is no customer identifier, so no cohort, retention, or repeat-purchase analysis is
  possible. That is a limit of the dataset, not an omission.
- `Transaction ID` uniqueness is assumed and never verified; a production pipeline would assert it.
- With no cost data, none of the margin questions a cafe actually cares about can be answered —
  every KPI here is a revenue proxy.

**Companion notebook:** the e-commerce exercise covers Structured Streaming, Auto Loader,
checkpointing, and event-time watermarks. Together the two cover both processing paradigms.